# 04 · Backtest: la cuarentena como experimento natural

*Segundos contra vidas* · Hackathon de Datos 2026 · ITM

**Pregunta:** cuando Medellín se vació (20 mar – 31 ago 2020) y se aceleró, ¿murieron más víctimas de las que se esperaban según quiénes fueron?

La exploración mostró que la letalidad subió (0,78% → 0,95%), pero sin corregir por la composición: en la cuarentena hubo menos peatones, más motos y otras horas. Aquí:
1. **Contrafactual:** el modelo logístico validado (D24), entrenado con 2014–2019, predice los muertos esperados según el perfil de cada víctima. Se compara con los observados (**O/E**).
2. **Placebo:** lo mismo para los mismos meses de 2019, sin cuarentena. Debe dar O/E ≈ 1.
3. **Exponente implícito:** con el aumento de velocidad de Google Traffic, ¿qué exponente implica el exceso? Se compara con Nilsson.
4. **Explicación rival:** hospitales saturados (capa del Distrito).

Es a nivel ciudad: no depende de los corredores. Cada parte: código → conclusión → decisión.

## 0 · Configuración

In [ ]:
import os, sys, unicodedata, re
from pathlib import Path

raiz = Path.cwd()
while not (raiz / "src" / "fuentes.py").exists():
    if raiz.parent == raiz:
        raise FileNotFoundError(f"No encontré el proyecto. El notebook está en: {Path.cwd()}")
    raiz = raiz.parent
os.chdir(raiz)
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from src.descarga import leer
from src.velocidad import tiempo_viaje_gt

pd.set_option("display.max_columns", 40)
Path("data/processed").mkdir(parents=True, exist_ok=True)

INICIO, FIN = pd.Timestamp("2020-03-20"), pd.Timestamp("2020-08-31")      # cuarentena (igual que en 01 y 02)
RAPIDAS = ["AV_REGIONAL", "AUT_NORTE", "AUT_SUR", "AV_PALMAS"]            # D18
print("Proyecto:", raiz)

In [ ]:
# Nombres de columnas uniformes: minúscula, sin tildes, con guion bajo
def _arreglar_texto(s):
    for codif in ("latin-1", "cp1252"):
        try:
            return s.encode(codif).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return s

def normalizar_columnas(df):
    nuevas = []
    for c in df.columns:
        c = unicodedata.normalize("NFKD", _arreglar_texto(str(c))).encode("ascii", "ignore").decode()
        nuevas.append(re.sub(r"[^0-9a-z]+", "_", c.lower()).strip("_"))
    df.columns = nuevas
    return df

def misma_ventana(fechas, ano):
    """True si la fecha cae entre el 20 de marzo y el 31 de agosto del año indicado."""
    return fechas.between(INICIO.replace(year=ano), FIN.replace(year=ano))

def o_sobre_e(obs, esp):
    """O/E con IC 95% exacto de Poisson para O, y p de la prueba O ~ Poisson(E)."""
    lo = stats.chi2.ppf(0.025, 2 * obs) / 2 if obs > 0 else 0.0
    hi = stats.chi2.ppf(0.975, 2 * (obs + 1)) / 2
    p = min(1.0, 2 * min(stats.poisson.cdf(obs, esp), stats.poisson.sf(obs - 1, esp)))
    return {"observados": int(obs), "esperados": esp, "O/E": obs / esp,
            "IC95_inf": lo / esp, "IC95_sup": hi / esp, "p": p}

## 1 · ¿Cuánto se aceleró la ciudad?

#### 1.1 Velocidad en la cuarentena frente a los mismos meses de 2017–2019 (Google Traffic)
Se comparan las mismas fechas del año (20 mar – 31 ago) para no mezclar la estacionalidad. La razón se calcula por corredor y luego se promedia en escala log (media geométrica).

In [ ]:
tv = tiempo_viaje_gt()
tv["fecha"] = pd.to_datetime(tv["fecha"])
tv["ano"] = tv["fecha"].dt.year
tv = tv[tv["velocidad_gt_kmh"] > 0]

en_ventana = pd.concat([misma_ventana(tv["fecha"], a) for a in range(2017, 2021)], axis=1).any(axis=1)
v = tv[en_ventana].assign(periodo=np.where(tv.loc[en_ventana, "ano"] == 2020, "cuarentena", "2017_2019"))
por_corr = v.groupby(["corredor_id", "periodo"])["velocidad_gt_kmh"].mean().unstack()
por_corr["razon"] = por_corr["cuarentena"] / por_corr["2017_2019"]
por_corr["tipo"] = np.where(por_corr.index.isin(RAPIDAS), "vías rápidas", "arterias")

def media_geo(s):
    return float(np.exp(np.log(s).mean()))

RAZON = {"todos": media_geo(por_corr["razon"]),
         "vías rápidas": media_geo(por_corr.loc[por_corr["tipo"] == "vías rápidas", "razon"]),
         "arterias": media_geo(por_corr.loc[por_corr["tipo"] == "arterias", "razon"])}
for k, r in RAZON.items():
    print(f"Velocidad en la cuarentena, {k}: ×{r:.3f} ({(r - 1) * 100:+.0f}%)")
por_corr.round(2).sort_values("razon", ascending=False)

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 2 · Contrafactual: ¿murieron más de los esperados?

#### 2.1 Víctimas (misma preparación que el notebook 02, D12)

In [ ]:
vic = normalizar_columnas(leer("victimas_medata"))
vic = vic[vic["ano"].notna()].copy()

vic["fecha"] = pd.to_datetime(vic["fecha_incidente"], dayfirst=True, errors="coerce")
vic["hora"] = pd.to_numeric(vic["hora"], errors="coerce")
vic["edad"] = pd.to_numeric(vic["edad"], errors="coerce")
vic["ano"] = vic["ano"].astype(int)
vic["muerto"] = (vic["gravedad_victima"] == "Muertos").astype(int)
vic["condicion"] = vic["condicion"].replace({"Acompañante de Motocicleta": "Acompañante de motocicleta"})
vic["actor"] = vic["condicion"].replace({"Motociclista": "Moto", "Acompañante de motocicleta": "Moto"})

vic["grupo_edad"] = pd.cut(vic["edad"], [0, 19, 29, 44, 59, 120], labels=["<20", "20-29", "30-44", "45-59", "60+"])
vic["franja"] = pd.cut(vic["hora"], [-1, 5, 17, 23], labels=["madrugada (0-5)", "dia (6-17)", "noche (18-23)"])
vic["fin_de_semana"] = (vic["fecha"].dt.dayofweek >= 5).astype(int)

usadas = ["muerto", "actor", "grupo_edad", "sexo", "franja", "fin_de_semana", "fecha"]
m = vic[vic["sexo"].isin(["M", "F"])].dropna(subset=usadas)[usadas + ["ano", "hora"]].copy()
print(f"Víctimas para el backtest: {len(m):,} · muertos: {m['muerto'].sum():,}")
print(f"De ellas en la cuarentena: {m['fecha'].between(INICIO, FIN).sum():,}")

#### 2.2 Muertos observados contra esperados
El modelo es el de la validación del notebook 02 (sin año ni cuarentena, entrenado con 2014–2019). Si **O/E > 1**, murieron más víctimas de las que explica su perfil.

In [ ]:
formula = ("muerto ~ C(actor, Treatment('Moto')) + C(grupo_edad, Treatment('20-29'))"
           " + C(sexo, Treatment('M')) + C(franja, Treatment('dia (6-17)')) + fin_de_semana")
modelo = smf.logit(formula, data=m[m["ano"] <= 2019]).fit(disp=0)

periodos = {
    "2020 antes (1 ene – 19 mar)":     m["fecha"].between("2020-01-01", "2020-03-19"),
    "Cuarentena (20 mar – 31 ago)":    m["fecha"].between(INICIO, FIN),
    "Reapertura (sep – dic 2020)":     m["fecha"].between("2020-09-01", "2020-12-31"),
    "2021 (ene – sep)":                m["fecha"].between("2021-01-01", "2021-09-30"),
}
filas = []
for nombre, filtro in periodos.items():
    d = m[filtro]
    filas.append({"periodo": nombre, "victimas": len(d), **o_sobre_e(d["muerto"].sum(), modelo.predict(d).sum())})
oe = pd.DataFrame(filas).set_index("periodo")
oe.round(3)

#### 2.3 Placebos: los mismos meses de 2015 a 2019
Para cada año sin cuarentena se entrena con los demás años de 2014–2019 y se predice su 20 mar – 31 ago. Si el método está bien, estos O/E rondan 1: muestran cuánto varía el O/E en un año normal.

In [ ]:
placebos, mods = [], {}
for ano in range(2015, 2020):
    mods[ano] = smf.logit(formula, data=m[(m["ano"] <= 2019) & (m["ano"] != ano)]).fit(disp=0)
    d = m[misma_ventana(m["fecha"], ano)]
    placebos.append({"periodo": f"Placebo {ano}", "victimas": len(d),
                     **o_sobre_e(d["muerto"].sum(), mods[ano].predict(d).sum())})
placebo = pd.DataFrame(placebos).set_index("periodo")
print(f"Rango de O/E en años normales: {placebo['O/E'].min():.2f} a {placebo['O/E'].max():.2f}")
print(f"¿La cuarentena queda por encima de todos los placebos? {oe.loc['Cuarentena (20 mar – 31 ago)', 'O/E'] > placebo['O/E'].max()}")
placebo.round(3)

#### 2.4 Por actor (moto y peatón)
La composición cambió en la cuarentena (menos peatones), por eso se mira cada actor por separado (D19).

In [ ]:
filas = []
for actor in ["Moto", "Peatón"]:
    obs = esp = n = 0
    for ano, mod in mods.items():
        d = m[misma_ventana(m["fecha"], ano) & (m["actor"] == actor)]
        obs, esp, n = obs + d["muerto"].sum(), esp + mod.predict(d).sum(), n + len(d)
    filas.append({"actor": actor, "periodo": "Placebos 2015–2019", "victimas": n, **o_sobre_e(obs, esp)})
    d = m[m["fecha"].between(INICIO, FIN) & (m["actor"] == actor)]
    filas.append({"actor": actor, "periodo": "Cuarentena 2020", "victimas": len(d),
                  **o_sobre_e(d["muerto"].sum(), modelo.predict(d).sum())})
oe_actor = pd.DataFrame(filas).set_index(["actor", "periodo"])
oe_actor.round(3)

#### 2.5 Sensibilidad: solo de 6:00 a 20:00
Es la ventana que mide Google Traffic. Así la velocidad y las víctimas cubren las mismas horas.

In [ ]:
dia = m["hora"].between(6, 20)
obs = sum(m[misma_ventana(m["fecha"], a) & dia]["muerto"].sum() for a in mods)
esp = sum(mod.predict(m[misma_ventana(m["fecha"], a) & dia]).sum() for a, mod in mods.items())
cuar_dia = m[m["fecha"].between(INICIO, FIN) & dia]
sens = pd.DataFrame([
    {"periodo": "Placebos 2015–2019 · 6–20 h", **o_sobre_e(obs, esp)},
    {"periodo": "Cuarentena · 6–20 h", **o_sobre_e(cuar_dia["muerto"].sum(), modelo.predict(cuar_dia).sum())},
]).set_index("periodo")
sens.round(3)

#### 2.6 Gráfica

In [ ]:
g = pd.concat([placebo, oe])
fig, ax = plt.subplots(figsize=(9, 3.8))
colores = ["#7f8c8d" if ("Placebo" in i or "Cuarentena" not in i) else "#c0392b" for i in g.index]
for k, (i, r) in enumerate(g.iterrows()):
    ax.plot([r["IC95_inf"], r["IC95_sup"]], [k, k], color=colores[k], lw=3, alpha=0.5)
    ax.scatter(r["O/E"], k, color=colores[k], s=60, zorder=3)
    ax.text(r["IC95_sup"] + 0.02, k, f"{r['O/E']:.2f}  ({r['observados']} vs {r['esperados']:.0f})", va="center", fontsize=9)
ax.axvline(1, ls="--", color="black", lw=1)
ax.set_yticks(range(len(g)), g.index); ax.invert_yaxis()
ax.set_xlabel("Muertos observados / esperados según el perfil de las víctimas (IC 95%)")
ax.set_title("¿Murieron más de lo esperado cuando la ciudad se aceleró?")
plt.tight_layout(); plt.show()

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 3 · ¿Qué exponente implica?
Nilsson: los siniestros con heridos crecen con la velocidad a la ~2 y los mortales a la ~4. Entonces la proporción de víctimas que muere crece a la **~2** (4 − 2).

Si la velocidad se multiplicó por *r*, Nilsson predice **O/E ≈ r²**. Al revés, el exponente implícito de Medellín es **β = ln(O/E) / ln(r)**.

In [ ]:
def implicito(fila, r):
    return {"β implícito": np.log(fila["O/E"]) / np.log(r),
            "β IC95_inf": np.log(max(fila["IC95_inf"], 1e-9)) / np.log(r),
            "β IC95_sup": np.log(fila["IC95_sup"]) / np.log(r),
            "O/E observado": fila["O/E"], "O/E si Nilsson (r²)": r ** 2}

c = oe.loc["Cuarentena (20 mar – 31 ago)"]
c_dia = sens.loc["Cuarentena · 6–20 h"]
filas = [{"víctimas": "todas", "velocidad": k, "r": r, **implicito(c, r)} for k, r in RAZON.items()]
filas += [{"víctimas": "6–20 h", "velocidad": "todos", "r": RAZON["todos"], **implicito(c_dia, RAZON["todos"])}]
filas += [{"víctimas": a, "velocidad": "todos", "r": RAZON["todos"],
           **implicito(oe_actor.loc[(a, "Cuarentena 2020")], RAZON["todos"])} for a in ["Moto", "Peatón"]]
exp_impl = pd.DataFrame(filas).set_index(["víctimas", "velocidad"])
exp_impl.round(2)

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 4 · Explicación rival: ¿hospitales saturados?
Si en la cuarentena murieron más heridos porque los hospitales estaban llenos, debería subir la proporción de muertes en **centros asistenciales**. Las muertes **en la vía** no dependen del hospital.

La capa del Distrito empieza en 2019: se comparan los mismos meses (20 mar – 31 ago) de cada año.

In [ ]:
mort = normalizar_columnas(leer("mortalidad_distrito"))
mort["fecha"] = pd.to_datetime(mort["fecha_ocurrencia"], dayfirst=True, errors="coerce")
mort["ano"] = mort["fecha"].dt.year
mort["lugar"] = mort["lugar_inspeccion"].astype(str).str.strip().replace(
    {"Vía Pública": "en la vía", "Centro Asistencial": "centro asistencial"})
mort.loc[~mort["lugar"].isin(["en la vía", "centro asistencial"]), "lugar"] = "otro / sin dato"

ventana = pd.concat([misma_ventana(mort["fecha"], a) for a in range(2019, 2025)], axis=1).any(axis=1)
w = mort[ventana]
lugar = pd.crosstab(w["ano"], w["lugar"])
lugar["total"] = lugar.sum(axis=1)
lugar["% en centro asistencial"] = (lugar.get("centro asistencial", 0) / lugar["total"] * 100).round(1)

cuar = lugar.loc[2020]
resto = lugar.drop(index=2020)[["centro asistencial", "total"]].sum()
tabla = [[cuar["centro asistencial"], cuar["total"] - cuar["centro asistencial"]],
         [resto["centro asistencial"], resto["total"] - resto["centro asistencial"]]]
_, p_fisher = stats.fisher_exact(tabla)
print(f"% en centro asistencial · 2020: {cuar['centro asistencial'] / cuar['total'] * 100:.1f}% "
      f"· otros años: {resto['centro asistencial'] / resto['total'] * 100:.1f}% · p (Fisher) = {p_fisher:.3f}")
lugar

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 5 · Resumen
_(completar al final)_

- ¿Murieron más víctimas de las esperadas en la cuarentena, una vez corregida la composición? (O/E, IC y p).
- ¿El placebo da ≈ 1? (si no, el método tiene un sesgo y hay que decirlo).
- ¿Dónde está el exceso: moto, peatón, de día?
- Exponente implícito frente a Nilsson (≈ 2), y frente al exponente del notebook 03 cuando esté.
- ¿Se descarta la explicación de los hospitales?
- Limitaciones: velocidad de Google Traffic (12 corredores, 6–20 h) como proxy de toda la ciudad; pocos muertos en la cuarentena; también cambiaron otras cosas (alcohol, tipo de viaje, toque de queda).